<a href="https://colab.research.google.com/github/nellypoghosyan/ML-FlyRank-AI/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

**Lane:** Refresh / Content Opportunity Scoring  
**Development month:** March 2026 (`month=2026-03`)  
**Decision moment:** end of **2026-03-21**  
**Feature window:** **2026-03-01 → 2026-03-21**  
**Outcome window:** **2026-03-22 → 2026-03-31**

I use a mid-panel month rather than the final June `_sample`, so the final month remains sealed for later validation.

The goal is decision support: rank content pages for human review based on signals that are genuinely knowable before the decision moment.

## 1. Unit of analysis + time window

### Five contract answers, in plain words

1. **What one row means:** one pseudonymized content item for one pseudonymized client at the March 21 decision moment, after aggregating its daily observations over the feature window.

2. **Tables I use:** `fact_content_daily_performance` for daily search/analytics signals. I keep `client_hash_id` and `content_hash_id` only as context keys.

3. **Time window:** features come only from **March 1–21, 2026**. The outcome is measured later, on **March 22–31, 2026**.

4. **What I predict / rank:** a binary proxy called `future_decline_label`. A page is positive when its average daily GSC impressions in the future 10-day outcome window are **at least 20% lower** than its average daily impressions in the 21-day feature window. I require at least 100 feature-window impressions so tiny pages do not dominate the label.

5. **One thing I deliberately exclude:** any future-window measurement, including `future_avg_daily_impressions` and `future_drop_ratio`. Those columns are computed from data after the decision moment, so they are label-derived and would leak the answer.

In [1]:
# Colab setup: read HF_TOKEN from Secrets. Never paste the token into this notebook.
!pip -q install duckdb

import duckdb
import pandas as pd
import numpy as np
from IPython.display import display

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    raise RuntimeError(
        "Open this notebook in Colab and add a Secret named HF_TOKEN first."
    ) from e

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN secret is missing.")

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE huggingface, TOKEN '{safe_token}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"{BASE}/fact_content_daily_performance/month=2026-03/*.parquet"

FEATURE_START = "2026-03-01"
DECISION_DATE = "2026-03-21"
OUTCOME_START = "2026-03-22"
OUTCOME_END = "2026-03-31"

print("Connected. Development partition:", MARCH)
print("Feature window:", FEATURE_START, "to", DECISION_DATE)
print("Outcome window:", OUTCOME_START, "to", OUTCOME_END)

Connected. Development partition: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet
Feature window: 2026-03-01 to 2026-03-21
Outcome window: 2026-03-22 to 2026-03-31


## 2. Fields: feature / label / context / excluded

### Feature candidates — maximum five
- `log_impressions_21d`
- `ctr_21d`
- `avg_position_21d`
- `days_with_impressions_21d`
- `log_ga4_sessions_21d`

### Label / proxy
- `future_decline_label`

### Context only
- `client_hash_id`
- `content_hash_id`

### Excluded
- `future_avg_daily_impressions` — measured after the decision moment.
- `future_drop_ratio` — directly used to define the label; deliberate leakage demo only, then removed.
- Raw/private identifiers — not present in the release and never needed for this lane.
- `client_hash_id` / `content_hash_id` as model inputs — pseudonymous IDs are useful for joins/splits, not predictive features.

## 3. Verify it with exactly three verification queries

These are the three contract checks required by the assignment.

### Verification query 1 — grain
The warehouse fact grain should be one row per `report_date × client_hash_id × content_hash_id`.  
If this query returns **zero rows**, the claimed grain holds for March 2026.

In [2]:
# VERIFICATION QUERY 1 OF 3 — grain
q1 = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS n
FROM read_parquet('{MARCH}')
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 10
"""
grain_check = con.sql(q1).df()
display(grain_check)

if grain_check.empty:
    print("PASS: zero duplicate grain groups returned.")
else:
    print("REVIEW: duplicate grain groups exist.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,n


PASS: zero duplicate grain groups returned.


### Verification query 2 — slice row count + date span
This checks how many daily rows are in the March partition and proves its observed date window.

In [3]:
# VERIFICATION QUERY 2 OF 3 — count + date span
q2 = f"""
SELECT
    COUNT(*) AS march_rows,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items
FROM read_parquet('{MARCH}')
"""
slice_check = con.sql(q2).df()
display(slice_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,min_report_date,max_report_date,clients,content_items
0,9841378,2026-03-01,2026-03-31,55,331437


### Verification query 3 — GA4 availability
Rows from before a client's GA4 tracking start can be zero-filled, so a zero is not automatically “no engagement.”  
I explicitly filter with **`ga4_data_available IS TRUE`** and show how many March rows survive.

In [4]:
# VERIFICATION QUERY 3 OF 3 — availability with IS TRUE
q3 = f"""
SELECT
    COUNT(*) AS total_march_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*),
        2
    ) AS pct_rows_surviving
FROM read_parquet('{MARCH}')
"""
availability_check = con.sql(q3).df()
display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_march_rows,ga4_available_rows,pct_rows_surviving
0,9841378,413966,4.21


## Five-feature frame

The frame is one row per content item/client at the March 21 decision moment.

**Available when?**
1. `log_impressions_21d` — knowable at the decision moment because it uses only GSC impressions observed from March 1–21.
2. `ctr_21d` — knowable at the decision moment because both clicks and impressions are accumulated only through March 21.
3. `avg_position_21d` — knowable at the decision moment because it averages only pre-decision GSC position observations.
4. `days_with_impressions_21d` — knowable at the decision moment because it counts pre-decision days with observed search exposure.
5. `log_ga4_sessions_21d` — knowable at the decision moment because it sums sessions only where `ga4_data_available IS TRUE` in the feature window.

The label uses March 22–31 only and is never included in the honest feature set.

In [5]:
# Build the decision-grain feature frame and a future outcome.
# This is feature construction, not one of the three verification queries.

feature_sql = f"""
WITH per_item AS (
    SELECT
        client_hash_id,
        content_hash_id,

        -- Feature window: 2026-03-01 through 2026-03-21
        SUM(gsc_impressions) FILTER (
            WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{DECISION_DATE}'
        ) AS impressions_21d,

        SUM(gsc_clicks) FILTER (
            WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{DECISION_DATE}'
        ) AS clicks_21d,

        AVG(gsc_avg_position) FILTER (
            WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{DECISION_DATE}'
              AND gsc_impressions > 0
              AND gsc_avg_position > 0
        ) AS avg_position_21d,

        COUNT(*) FILTER (
            WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{DECISION_DATE}'
              AND gsc_impressions > 0
        ) AS days_with_impressions_21d,

        SUM(ga4_sessions) FILTER (
            WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{DECISION_DATE}'
              AND ga4_data_available IS TRUE
        ) AS ga4_sessions_21d,

        COUNT(*) FILTER (
            WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{DECISION_DATE}'
              AND ga4_data_available IS TRUE
        ) AS ga4_available_days_21d,

        -- Future outcome window: 2026-03-22 through 2026-03-31
        AVG(gsc_impressions) FILTER (
            WHERE report_date BETWEEN DATE '{OUTCOME_START}' AND DATE '{OUTCOME_END}'
        ) AS future_avg_daily_impressions,

        AVG(gsc_impressions) FILTER (
            WHERE report_date BETWEEN DATE '{FEATURE_START}' AND DATE '{DECISION_DATE}'
        ) AS past_avg_daily_impressions

    FROM read_parquet('{MARCH}')
    GROUP BY 1, 2
)
SELECT
    client_hash_id,
    content_hash_id,

    LN(1 + impressions_21d) AS log_impressions_21d,
    CASE WHEN impressions_21d > 0
         THEN 1.0 * clicks_21d / impressions_21d
    END AS ctr_21d,
    avg_position_21d,
    days_with_impressions_21d,
    LN(1 + COALESCE(ga4_sessions_21d, 0)) AS log_ga4_sessions_21d,

    -- Outcome helpers: retained for the label/leakage demonstration, NOT honest features.
    past_avg_daily_impressions,
    future_avg_daily_impressions,
    CASE
        WHEN past_avg_daily_impressions > 0
        THEN future_avg_daily_impressions / past_avg_daily_impressions
    END AS future_drop_ratio,

    CASE
        WHEN impressions_21d >= 100
         AND future_avg_daily_impressions IS NOT NULL
         AND past_avg_daily_impressions > 0
         AND future_avg_daily_impressions < 0.80 * past_avg_daily_impressions
        THEN 1
        ELSE 0
    END AS future_decline_label

FROM per_item
WHERE impressions_21d >= 100
  AND future_avg_daily_impressions IS NOT NULL
  AND past_avg_daily_impressions > 0
  AND ga4_available_days_21d > 0
"""

frame = con.sql(feature_sql).df()

HONEST_FEATURES = [
    "log_impressions_21d",
    "ctr_21d",
    "avg_position_21d",
    "days_with_impressions_21d",
    "log_ga4_sessions_21d",
]

print("Decision-grain rows:", len(frame))
print("Positive label rate:", round(frame["future_decline_label"].mean(), 4))
display(frame[["client_hash_id", "content_hash_id"] + HONEST_FEATURES + ["future_decline_label"]].head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Decision-grain rows: 36712
Positive label rate: 0.2711


,client_hash_id,content_hash_id,log_impressions_21d,ctr_21d,avg_position_21d,days_with_impressions_21d,log_ga4_sessions_21d,future_decline_label
0,client_65de48885f4ef01b,content_86cd9b9636f4fd46,5.978886,0.005076,5.630354,19,1.791759,0
1,client_65de48885f4ef01b,content_1982718dcce6d54e,6.549651,0.004298,5.654756,21,2.944439,0
2,client_65de48885f4ef01b,content_da2b299fbd22fe0d,6.150603,0.002137,13.807277,21,0.693147,1
3,client_65de48885f4ef01b,content_56f9a62f24be5548,5.823046,0.000000,13.219549,21,0.693147,1
4,client_65de48885f4ef01b,content_859fb78a92a7bb9d,5.739793,0.003226,20.978152,13,1.098612,0
5,client_65de48885f4ef01b,content_1ea99aa7351c760f,6.979145,0.000932,15.437191,21,1.098612,1
6,client_65de48885f4ef01b,content_e5b87303deb95692,6.822197,0.003272,14.075248,21,1.609438,0
7,client_65de48885f4ef01b,content_2822a079e6e1e9df,4.859812,0.000000,52.486254,21,0.693147,0
8,client_65de48885f4ef01b,content_f95a3b3cf73f8ef3,4.753590,0.008696,4.613887,19,1.098612,0
9,client_65de48885f4ef01b,content_f4dd61d41cf5376d,5.556828,0.011628,5.009724,21,1.609438,0


## The trap — deliberately add one label-derived column, then delete it

`future_drop_ratio` uses March 22–31 information. That information does not exist at the March 21 decision moment, and it directly determines the label threshold.

I add it **once on purpose** to demonstrate leakage, compare the score, then remove it.

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

# Keep rows with both classes represented.
model_df = frame.copy()

train_idx, test_idx = train_test_split(
    np.arange(len(model_df)),
    test_size=0.30,
    random_state=42,
    stratify=model_df["future_decline_label"]
)

y_train = model_df.iloc[train_idx]["future_decline_label"]
y_test = model_df.iloc[test_idx]["future_decline_label"]

def quick_score(feature_cols):
    pipe = make_pipeline(
        SimpleImputer(strategy="median"),
        DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=42)
    )
    pipe.fit(model_df.iloc[train_idx][feature_cols], y_train)
    p = pipe.predict_proba(model_df.iloc[test_idx][feature_cols])[:, 1]
    pred = (p >= 0.5).astype(int)
    return {
        "ROC_AUC": roc_auc_score(y_test, p),
        "Accuracy": accuracy_score(y_test, pred)
    }

honest_score = quick_score(HONEST_FEATURES)

LEAK_FEATURE = "future_drop_ratio"
leaky_score = quick_score(HONEST_FEATURES + [LEAK_FEATURE])

comparison = pd.DataFrame([
    {"version": "honest five features", **honest_score},
    {"version": "with future_drop_ratio LEAK", **leaky_score},
])

display(comparison)

print(
    "\nLeakage lesson: future_drop_ratio is computed from the outcome window and "
    "directly encodes the label rule, so the leaky score should jump dramatically."
)

# DELETE THE LEAK: this is the only feature list kept for later work.
FINAL_FEATURES = HONEST_FEATURES.copy()
assert LEAK_FEATURE not in FINAL_FEATURES
print("Final feature set:", FINAL_FEATURES)
print("Leak removed:", LEAK_FEATURE not in FINAL_FEATURES)

,version,ROC_AUC,Accuracy
0,honest five features,0.652492,0.730979
1,with future_drop_ratio LEAK,1.000000,0.999909



Leakage lesson: future_drop_ratio is computed from the outcome window and directly encodes the label rule, so the leaky score should jump dramatically.
Final feature set: ['log_impressions_21d', 'ctr_21d', 'avg_position_21d', 'days_with_impressions_21d', 'log_ga4_sessions_21d']
Leak removed: True


## 4. Data limits

**Named limitation:** this warehouse is an **unbalanced panel**. Different clients began GSC and GA4 tracking at different times, so missing early history does not mean zero traffic. In particular, GA4 values can be zero-filled before tracking starts, which is why this notebook checks `ga4_data_available` and uses `IS TRUE`.

A second practical limitation is that this observational data can support a ranked review queue, but it cannot prove that refreshing a page **causes** recovery. That would require an experiment or another causal design.

In [7]:
# Compact final summary generated from the real run.
summary = {
    "lane": "Refresh / Content Opportunity Scoring",
    "decision_date": DECISION_DATE,
    "feature_window": f"{FEATURE_START} to {DECISION_DATE}",
    "outcome_window": f"{OUTCOME_START} to {OUTCOME_END}",
    "decision_grain_rows": len(frame),
    "positive_label_rate": float(frame["future_decline_label"].mean()),
    "honest_roc_auc": float(honest_score["ROC_AUC"]),
    "leaky_roc_auc": float(leaky_score["ROC_AUC"]),
    "final_features": FINAL_FEATURES,
}
pd.Series(summary)

,0
lane,Refresh / Content Opportunity Scoring
decision_date,2026-03-21
feature_window,2026-03-01 to 2026-03-21
outcome_window,2026-03-22 to 2026-03-31
decision_grain_rows,36712
positive_label_rate,0.27111
honest_roc_auc,0.652492
leaky_roc_auc,1.0
final_features,"[log_impressions_21d, ctr_21d, avg_position_21..."


## 5. Self-check

After **Runtime → Run all**, confirm:

- [ ] Five plain-words contract answers are filled.
- [ ] Exactly three verification queries have visible outputs.
- [ ] Grain query returns zero duplicate groups.
- [ ] March row count and date span are visible.
- [ ] Availability uses `ga4_data_available IS TRUE` and shows how many rows survive.
- [ ] Feature frame contains exactly five honest features.
- [ ] Every feature has an “available when?” sentence.
- [ ] The deliberate `future_drop_ratio` leak makes the quick score jump.
- [ ] The leak is removed from `FINAL_FEATURES`.
- [ ] One limitation is named.
- [ ] No token, raw client names, URLs, or private queries are committed.
- [ ] Notebook is saved to `work/notebooks/w03_data_contract.ipynb` and committed.